# Radar Visualizations — Seeing Pulses in the Real World

This notebook puts you inside a radar scenario.
Every cell renders a **physical scene** (antenna, targets, pulses flying through space)
side-by-side with the **signal plots** the radar operator actually sees.
Sliders let you control waveform parameters and step through time so you can
watch cause and effect unfold.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 9,
})

c = 3e8  # speed of light [m/s]
print("Ready — c = 3 × 10⁸ m/s")

Ready — c = 3 × 10⁸ m/s


## 1 — Pulse Propagation Scene

A radar emits a rectangular pulse of width $\tau$ (PW). The pulse travels at $c$, hits a target at range $R$, and returns after

$$\Delta t = \frac{2R}{c} \;\Longrightarrow\; R = \frac{c\,\Delta t}{2}$$

The **top panel** is a bird's-eye scene: antenna on the left, target on the right, and the pulse packet moving through space.
The **bottom panel** shows exactly what the radar receiver records — a time axis with the Tx burst and (later) the echo.

Use the **time slider** to step through the event and watch the pulse travel out, bounce, and return.
Change **PW** and **PRI** to see how they affect the listening window and duty cycle.

In [2]:
out1 = widgets.Output()

w_R1 = widgets.FloatSlider(value=30, min=5, max=120, step=0.5,
    description="Target R [km]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))
w_pw1 = widgets.FloatSlider(value=5, min=1, max=30, step=0.5,
    description="PW τ [µs]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))
w_pri1 = widgets.FloatSlider(value=800, min=100, max=2000, step=50,
    description="PRI [µs]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))
w_t1 = widgets.FloatSlider(value=0, min=0, max=1.0, step=0.002,
    description="Time [ms]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"), readout_format=".3f")

def draw_scene1(change=None):
    R = w_R1.value * 1e3
    tau = w_pw1.value * 1e-6
    pri = w_pri1.value * 1e-6
    dt_rt = 2 * R / c
    t_view = max(dt_rt * 1.5, pri)
    w_t1.max = round(t_view * 1e3, 3)
    t_now = w_t1.value * 1e-3
    pulse_m = c * tau
    R_ua = c * pri / 2
    duty = tau / pri * 100

    # pulse packet position
    if t_now <= tau:
        tx_front = c * t_now
        tx_back = 0.0
        echo_front = echo_back = -1
        phase = "transmitting"
    elif t_now <= dt_rt / 2:
        tx_front = c * t_now
        tx_back = tx_front - pulse_m
        echo_front = echo_back = -1
        phase = "pulse outbound"
    elif t_now <= dt_rt / 2 + tau:
        tx_front = R
        tx_back = max(R - pulse_m, c * t_now - pulse_m) if c * t_now > R else c * t_now - pulse_m
        tx_front = min(c * t_now, R)
        tx_back = max(tx_front - pulse_m, 0)
        reflect_time = t_now - R / c
        echo_front = R - c * reflect_time
        echo_back = min(echo_front + pulse_m, R)
        phase = "reflecting"
    elif t_now <= dt_rt:
        tx_front = tx_back = -1
        echo_front = R - c * (t_now - R / c)
        echo_back = echo_front + pulse_m
        echo_front = max(echo_front, 0)
        echo_back = min(echo_back, R)
        phase = "echo returning"
    elif t_now <= dt_rt + tau:
        tx_front = tx_back = -1
        echo_front = 0
        echo_back = R - c * (t_now - R / c) + pulse_m
        echo_back = max(echo_back, 0)
        phase = "echo arriving"
    else:
        tx_front = tx_back = echo_front = echo_back = -1
        phase = "echo received"

    with out1:
        clear_output(wait=True)
        fig, (ax_s, ax_sig) = plt.subplots(2, 1, figsize=(11, 5.8),
            gridspec_kw={"height_ratios": [1.3, 1]})

        # scene
        ax_s.set_xlim(-R * 0.07, R * 1.12)
        ax_s.set_ylim(-1.3, 1.3)
        ax_s.set_yticks([])
        ax_s.axhline(0, color="gray", lw=0.3)
        ax_s.plot(0, 0, "s", color="#1f77b4", ms=20, zorder=5)
        ax_s.annotate("RADAR", (0, -0.75), ha="center", fontsize=9,
                      color="#1f77b4", weight="bold")
        ax_s.plot(R, 0, "D", color="#d62728", ms=16, zorder=5)
        ax_s.annotate(f"TARGET  {R/1e3:.1f} km", (R, -0.75),
                      ha="center", fontsize=9, color="#d62728")
        # Tx pulse packet
        if tx_front >= 0 and tx_back >= 0 and tx_front > tx_back:
            ax_s.axvspan(tx_back, tx_front, ymin=0.42, ymax=0.58,
                         color="#1f77b4", alpha=0.75)
            ax_s.annotate("Tx", ((tx_front + tx_back) / 2, 0.45),
                          fontsize=8, color="#1f77b4", ha="center")
        # echo packet
        if echo_front >= 0 and echo_back >= 0 and echo_back > echo_front:
            ax_s.axvspan(echo_front, echo_back, ymin=0.42, ymax=0.58,
                         color="#d62728", alpha=0.65)
            mid_e = (echo_front + echo_back) / 2
            ax_s.annotate("Echo", (mid_e, -0.45), fontsize=8,
                          color="#d62728", ha="center")
        ax_s.xaxis.set_major_formatter(
            plt.FuncFormatter(lambda x, _: f"{x/1e3:.0f}"))
        ax_s.set_xlabel("Distance [km]")
        ax_s.set_title(f"Scene — {phase}   |   PW = {tau*1e6:.1f} µs   "
                       f"PRI = {pri*1e6:.0f} µs   Duty = {duty:.2f}%   "
                       f"R_unamb = {R_ua/1e3:.1f} km", fontsize=10)

        # signal timeline
        t_axis = np.linspace(0, t_view, 5000)
        tx_env = np.where((t_axis >= 0) & (t_axis <= tau), 1.0, 0.0)
        rx_env = np.where((t_axis >= dt_rt) & (t_axis <= dt_rt + tau), 0.6, 0.0)
        ax_sig.fill_between(t_axis * 1e6, tx_env, alpha=0.4, color="#1f77b4",
                            label="Tx pulse")
        ax_sig.fill_between(t_axis * 1e6, rx_env, alpha=0.5, color="#d62728",
                            label="Echo")
        ax_sig.axvline(t_now * 1e6, color="k", ls="--", lw=1.2,
                       label="current time")
        # PRI bracket
        ax_sig.axvspan(0, pri * 1e6, ymin=0, ymax=0.05, color="#2ca02c",
                       alpha=0.3)
        ax_sig.text(pri * 1e6 / 2, -0.18, f"PRI = {pri*1e6:.0f} µs",
                    ha="center", fontsize=8, color="#2ca02c")
        # round-trip annotation
        if dt_rt * 1e6 < t_view * 1e6:
            ax_sig.annotate("", xy=(dt_rt * 1e6, 0.8), xytext=(0, 0.8),
                arrowprops=dict(arrowstyle="<->", color="k", lw=1.2))
            ax_sig.text(dt_rt * 1e6 / 2, 0.88,
                f"Δt = {dt_rt*1e6:.1f} µs → R = {R/1e3:.1f} km",
                ha="center", fontsize=8,
                bbox=dict(fc="white", ec="gray", pad=2))
        ax_sig.set_xlabel("Time [µs]")
        ax_sig.set_ylabel("Amplitude")
        ax_sig.set_ylim(-0.3, 1.15)
        ax_sig.set_title("Receiver timeline", fontsize=10)
        ax_sig.legend(loc="upper right", fontsize=8)
        plt.tight_layout()
        plt.show()

for w in [w_R1, w_pw1, w_pri1, w_t1]:
    w.observe(draw_scene1, "value")
display(widgets.VBox([w_R1, w_pw1, w_pri1, w_t1]), out1)
draw_scene1()

Output()

## 2 — Target Distinction Based on Pulse Width

Two targets at ranges $R_1$ and $R_2$ return echoes separated in time by
$\Delta t = 2(R_2 - R_1)/c$. Each echo lasts as long as the transmitted pulse $\tau$.
The radar can distinguish the two targets **only** when their echoes do not overlap:

$$\Delta R = R_2 - R_1 > \frac{c\,\tau}{2}$$

A shorter pulse gives finer resolution but carries less energy.

The **top panel** shows the scene: radar, two targets, and the pulse packet travelling through space.
The **bottom panel** shows the receiver timeline — when echoes overlap they merge into one blob
and the radar cannot tell two objects apart. Use the **time slider** to animate and
**PW** / **separation** sliders to cross the resolution threshold.

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})
c = 3e8

out = widgets.Output()

w_R = widgets.FloatSlider(value=30, min=5, max=80, step=0.5,
    description="Target 1 R [km]:", style={"description_width": "130px"},
    layout=widgets.Layout(width="600px"))
w_sep = widgets.FloatSlider(value=1500, min=50, max=5000, step=50,
    description="Separation [m]:", style={"description_width": "130px"},
    layout=widgets.Layout(width="600px"))
w_pw = widgets.FloatSlider(value=10, min=1, max=40, step=0.5,
    description="PW τ [µs]:", style={"description_width": "130px"},
    layout=widgets.Layout(width="600px"))
w_t = widgets.FloatSlider(value=0, min=0, max=1.0, step=0.002,
    description="Time [ms]:", style={"description_width": "130px"},
    layout=widgets.Layout(width="600px"), readout_format=".3f")

def draw(change=None):
    R1 = w_R.value * 1e3
    sep = w_sep.value
    R2 = R1 + sep
    tau = w_pw.value * 1e-6
    pulse_m = c * tau
    min_sep = c * tau / 2
    resolved = sep >= min_sep
    dt1 = 2 * R1 / c
    dt2 = 2 * R2 / c
    t_view = dt2 * 1.5
    w_t.max = round(t_view * 1e3, 3)
    t_now = w_t.value * 1e-3

    # pulse position helpers
    def pulse_pos(t, R_tgt):
        """Return (front, back) of pulse in metres; -1 if gone."""
        t_hit = R_tgt / c
        if t <= 0:
            return (-1, -1)
        if t <= t_hit:
            front = min(c * t, R_tgt)
            back = max(front - pulse_m, 0)
            return (back, front)
        t_back = t - t_hit
        front = R_tgt - c * t_back
        back = front + pulse_m
        if back < 0:
            return (-1, -1)
        return (max(front, 0), min(back, R_tgt))

    clr = "#2ca02c" if resolved else "#d62728"
    tag = "RESOLVED" if resolved else "NOT RESOLVED"

    with out:
        clear_output(wait=True)
        fig, (ax_s, ax_sig) = plt.subplots(2, 1, figsize=(12, 6.2),
            gridspec_kw={"height_ratios": [1.3, 1]})

        # === scene ===
        margin = R2 * 0.08
        ax_s.set_xlim(-margin, R2 + margin)
        ax_s.set_ylim(-1.6, 1.6)
        ax_s.set_yticks([])
        ax_s.axhline(0, color="gray", lw=0.3)

        # radar
        ax_s.plot(0, 0, "s", color="#1f77b4", ms=18, zorder=5)
        ax_s.annotate("RADAR", (0, -0.9), ha="center",
                      fontsize=9, color="#1f77b4", weight="bold")

        # targets
        ax_s.plot(R1, 0, "D", color="#ff7f0e", ms=14, zorder=5)
        ax_s.annotate(f"T1  {R1/1e3:.1f} km", (R1, 0.6),
                      ha="center", fontsize=9, color="#ff7f0e")
        ax_s.plot(R2, 0, "D", color="#9467bd", ms=14, zorder=5)
        ax_s.annotate(f"T2  {R2/1e3:.1f} km", (R2, 0.6),
                      ha="center", fontsize=9, color="#9467bd")

        # separation bracket
        ax_s.annotate("", xy=(R2, -0.55), xytext=(R1, -0.55),
            arrowprops=dict(arrowstyle="<->", color="k", lw=1.3))
        ax_s.text((R1 + R2) / 2, -0.85,
            f"Δ = {sep:.0f} m   (min = {min_sep:.0f} m)",
            ha="center", fontsize=9,
            bbox=dict(fc="lightyellow", ec="gray", pad=3))

        # outgoing pulse (before it splits on targets)
        if t_now <= R1 / c:
            pf = min(c * t_now, R1)
            pb = max(pf - pulse_m, 0)
            if pf > pb:
                ax_s.axvspan(pb, pf, ymin=0.42, ymax=0.58,
                             color="#1f77b4", alpha=0.7)
                ax_s.annotate("Tx", ((pf + pb) / 2, 0.35),
                              fontsize=8, color="#1f77b4", ha="center")
        else:
            # echo from T1
            b1, f1 = pulse_pos(t_now, R1)
            if b1 >= 0 and f1 > b1:
                ax_s.axvspan(b1, f1, ymin=0.42, ymax=0.58,
                             color="#ff7f0e", alpha=0.6)
                ax_s.annotate("Echo1", ((b1 + f1) / 2, -0.35),
                              fontsize=7, color="#ff7f0e", ha="center")
            # portion still going toward T2 or echo from T2
            b2, f2 = pulse_pos(t_now, R2)
            if b2 >= 0 and f2 > b2:
                cc = "#9467bd" if t_now > R2 / c else "#1f77b4"
                ax_s.axvspan(b2, f2, ymin=0.42, ymax=0.58,
                             color=cc, alpha=0.6)
                lbl = "Echo2" if t_now > R2 / c else "Tx"
                ax_s.annotate(lbl, ((b2 + f2) / 2, 0.35 if lbl == "Tx" else -0.35),
                              fontsize=7, color=cc, ha="center")

        ax_s.xaxis.set_major_formatter(
            plt.FuncFormatter(lambda x, _: f"{x/1e3:.0f}"))
        ax_s.set_xlabel("Distance [km]")
        ax_s.set_title(f"Scene  —  [{tag}]   PW = {tau*1e6:.1f} µs   "
                       f"min ΔR = cτ/2 = {min_sep:.0f} m",
                       fontsize=10, color=clr)

        # === receiver timeline ===
        t_axis = np.linspace(0, t_view, 6000)
        tx_env = np.where((t_axis >= 0) & (t_axis <= tau), 1.0, 0.0)
        e1 = np.where((t_axis >= dt1) & (t_axis <= dt1 + tau), 0.7, 0.0)
        e2 = np.where((t_axis >= dt2) & (t_axis <= dt2 + tau), 0.6, 0.0)
        combined = np.maximum(e1, e2)

        ax_sig.fill_between(t_axis * 1e6, tx_env, alpha=0.3,
                            color="#1f77b4", label="Tx")
        ax_sig.fill_between(t_axis * 1e6, e1, alpha=0.4,
                            color="#ff7f0e", label="Echo T1")
        ax_sig.fill_between(t_axis * 1e6, e2, alpha=0.4,
                            color="#9467bd", label="Echo T2")
        ax_sig.plot(t_axis * 1e6, np.maximum(e1, e2), color=clr,
                    lw=1.8, label="Combined envelope")
        ax_sig.axvline(t_now * 1e6, color="k", ls="--", lw=1.2,
                       label="current time")

        # show overlap zone if not resolved
        if not resolved and dt2 < dt1 + tau:
            ax_sig.axvspan(dt2 * 1e6, (dt1 + tau) * 1e6,
                           alpha=0.15, color="#d62728")
            ax_sig.text((dt2 * 1e6 + (dt1 + tau) * 1e6) / 2, 0.9,
                        "OVERLAP", ha="center", fontsize=9,
                        color="#d62728", weight="bold")

        ax_sig.set_xlabel("Time [µs]")
        ax_sig.set_ylabel("Amplitude")
        ax_sig.set_ylim(-0.1, 1.15)
        ax_sig.set_title("Receiver timeline — can the radar separate the two echoes?",
                         fontsize=10)
        ax_sig.legend(loc="upper right", fontsize=7, ncol=3)
        plt.tight_layout()
        plt.show()

for w in [w_R, w_sep, w_pw, w_t]:
    w.observe(draw, "value")
display(widgets.VBox([w_R, w_sep, w_pw, w_t]), out)
draw()

Output()

## 3 — Multi-Pulse Train with Moving Target

Real radars fire a **train** of pulses at repetition interval PRI = 1/PRF.
Each pulse travels out, bounces, and returns — exactly like Section 1 but repeated.
If the target moves between pulses the echo shifts to a different time slot:

$$R_n = R_0 + v_r \cdot n \cdot \text{PRI}$$

**Top panel** — the scene for one pulse: use **Pulse #** to select which pulse,
then drag **Pulse phase** from 0→1 to watch that pulse fly out and the echo return.

**Bottom panel** — the **Range–Time Intensity** (RTI) map stacks all receive windows.
Each row is one pulse; the bright pixel is where the echo landed.
A moving target draws a **diagonal** track — its slope reveals the velocity.

In [ ]:
out2 = widgets.Output()

w_R2 = widgets.FloatSlider(value=40, min=5, max=100, step=1,
    description="Initial R [km]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))
w_v2 = widgets.FloatSlider(value=250, min=-500, max=500, step=10,
    description="Velocity [m/s]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))
w_prf2 = widgets.FloatSlider(value=1000, min=200, max=5000, step=100,
    description="PRF [Hz]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))
w_pulse2 = widgets.IntSlider(value=0, min=0, max=29, step=1,
    description="Pulse #:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))
w_phase2 = widgets.FloatSlider(value=0, min=0, max=1, step=0.01,
    description="Pulse phase:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"),
    readout_format=".2f")

def draw_rti(change=None):
    R0 = w_R2.value * 1e3
    vr = w_v2.value
    prf = w_prf2.value
    pri = 1.0 / prf
    N = 30
    p_idx = w_pulse2.value
    phase = w_phase2.value
    R_ua = c * pri / 2
    tau = 5e-6
    pulse_m = c * tau

    ranges = np.array([R0 + vr * i * pri for i in range(N)])
    R_now = ranges[p_idx]
    dt_rt = 2 * R_now / c
    t_now = phase * pri

    with out2:
        clear_output(wait=True)
        fig, axes = plt.subplots(2, 1, figsize=(12, 6.5),
            gridspec_kw={"height_ratios": [1.2, 1.3]})

        ax_s = axes[0]
        x_hi = min(R_ua, max(R0 + 20e3, R_now + 10e3))
        ax_s.set_xlim(-x_hi * 0.05, x_hi)
        ax_s.set_ylim(-1.5, 1.5)
        ax_s.set_yticks([])
        ax_s.axhline(0, color="gray", lw=0.3)

        # radar
        ax_s.plot(0, 0, "s", color="#1f77b4", ms=18, zorder=5)
        ax_s.annotate("RADAR", (0, -0.9), ha="center",
                      fontsize=9, color="#1f77b4", weight="bold")

        # target
        R_draw = np.clip(R_now, 500, x_hi * 0.95)
        tclr = "#2ca02c" if vr >= 0 else "#d62728"
        marker = "<" if vr > 0 else (">" if vr < 0 else "D")
        ax_s.plot(R_draw, 0, marker, color=tclr, ms=16, zorder=5)
        ax_s.annotate(f"TARGET  {R_now/1e3:.1f} km\nv = {vr:+.0f} m/s",
                      (R_draw, 0.55), ha="center", fontsize=9, color=tclr)

        # animate the pulse: outgoing then echo
        if t_now <= tau:
            pf = c * t_now
            pb = 0
            ax_s.axvspan(pb, pf, ymin=0.42, ymax=0.58, color="#1f77b4", alpha=0.7)
            ax_s.annotate("Tx", (pf / 2, 0.3), fontsize=8, color="#1f77b4", ha="center")
            status = "transmitting"
        elif t_now < R_now / c:
            pf = c * t_now
            pb = max(pf - pulse_m, 0)
            if pf > 0 and pf <= R_now:
                ax_s.axvspan(pb, pf, ymin=0.42, ymax=0.58, color="#1f77b4", alpha=0.7)
                ax_s.annotate("Tx →", (pf, 0.3), fontsize=8, color="#1f77b4", ha="center")
            status = "pulse outbound"
        elif t_now < dt_rt:
            ef = 2 * R_now - c * t_now
            eb = ef + pulse_m
            ef_c = max(ef, 0)
            eb_c = min(eb, R_now)
            if eb_c > ef_c:
                ax_s.axvspan(ef_c, eb_c, ymin=0.42, ymax=0.58, color="#ff7f0e", alpha=0.7)
                ax_s.annotate("← Echo", (ef_c, -0.4), fontsize=8, color="#ff7f0e", ha="center")
            status = "echo returning"
        elif t_now < dt_rt + tau:
            status = "echo arriving at receiver"
        else:
            status = "listening (waiting for next pulse)"

        ax_s.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e3:.0f}"))
        ax_s.set_xlabel("Range [km]")
        ax_s.set_title(f"Pulse #{p_idx} — {status}   |   "
                       f"R_ua = {R_ua/1e3:.1f} km   PRF = {prf:.0f} Hz",
                       fontsize=10)

        # receiver strip per pulse (RTI-style stacked)
        ax_rt = axes[1]
        n_show = min(N, 20)
        p_lo = max(0, p_idx - n_show + 1)
        r_bins = np.linspace(0, R_ua, 400)
        delta_r = c * tau / 2
        img = np.zeros((n_show, len(r_bins)))
        for k in range(n_show):
            ki = p_lo + k
            Ri = ranges[ki] if ki < N else ranges[-1]
            if 0 < Ri < R_ua:
                img[k] = np.exp(-0.5 * ((r_bins - Ri) / (delta_r * 0.4)) ** 2)

        extent = [r_bins[0] / 1e3, r_bins[-1] / 1e3, p_lo + n_show - 0.5, p_lo - 0.5]
        ax_rt.imshow(img, aspect="auto", extent=extent, cmap="inferno", vmin=0, vmax=1)
        ax_rt.axhline(p_idx, color="white", ls="--", lw=1, alpha=0.7)
        ax_rt.annotate(f"← pulse #{p_idx}", (R_ua / 1e3 * 0.85, p_idx),
                       color="white", fontsize=8, va="center")
        ax_rt.set_xlabel("Range [km]")
        ax_rt.set_ylabel("Pulse #")
        ax_rt.set_title("Range–Time Intensity: each row = one pulse's echo",
                        fontsize=10)
        plt.tight_layout()
        plt.show()

for w in [w_R2, w_v2, w_prf2, w_pulse2, w_phase2]:
    w.observe(draw_rti, "value")
display(widgets.VBox([w_R2, w_v2, w_prf2, w_pulse2, w_phase2]), out2)
draw_rti()

Output()

## 4 — Doppler Effect: Watching Wavefronts Compress and Stretch

A target moving at radial velocity $v_r$ toward the radar compresses the reflected wavefronts,
shifting the carrier frequency by the **Doppler shift**:

$$f_d = \frac{2\,v_r\,f_0}{c}$$

Positive $v_r$ (approaching) → higher frequency. Negative $v_r$ (receding) → lower frequency.

The **top panel** shows the scene: the radar transmits continuous wavefronts and the target
reflects them back. Approaching targets bunch the wavefronts together; receding ones spread them apart.
The **bottom panel** overlays the transmitted and received sinusoids so you can directly
see the frequency difference caused by the Doppler shift.

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

c = 3e8  # physical speed of light

out3 = widgets.Output()

w_v3 = widgets.FloatSlider(value=200, min=-500, max=500, step=5,
    description="Velocity [m/s]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))

w_f3 = widgets.FloatSlider(value=10, min=1, max=35, step=0.5,
    description="Carrier [GHz]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))

w_t3 = widgets.FloatSlider(value=0, min=0, max=4.0, step=0.05,
    description="Time [s]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))


def draw_doppler(change=None):

    vr = w_v3.value
    f0 = w_f3.value * 1e9
    lam = c / f0
    t_now = w_t3.value
    fd = 2 * vr * f0 / c

    R0 = 5000
    R_now = max(R0 - vr * t_now, 300)

    if vr > 0:
        label = "← approaching"
        clr = "#2ca02c"
    elif vr < 0:
        label = "receding →"
        clr = "#d62728"
    else:
        label = "stationary"
        clr = "gray"

    # visual propagation speed (scaled)
    c_vis = 2000

    # pulse emission parameters
    pulse_period = 0.15
    n_pulses = 60
    t_emits = np.arange(n_pulses) * pulse_period

    # keep pulses already emitted
    mask_emit = t_now > t_emits
    t_emits = t_emits[mask_emit]

    # outgoing pulse positions
    x_out = c_vis * (t_now - t_emits)

    # target position
    tgt_x = np.clip(R_now, 300, R0 * 1.1)

    # reflection time solving:
    # c(t_hit - t_emit) = R0 - v*t_hit
    t_hit = (R0 + c_vis * t_emits) / (c_vis + vr)

    # pulses that already hit target
    mask_hit = t_now > t_hit

    # reflection position
    x_hit = c_vis * (t_hit - t_emits)

    # echo propagation
    x_echo = x_hit - c_vis * (t_now - t_hit)

    valid_echo = (x_echo > 0) & mask_hit
    x_echo = x_echo[valid_echo]

    with out3:
        clear_output(wait=True)

        fig, (ax_s, ax_w) = plt.subplots(
            2, 1, figsize=(11, 6.2),
            gridspec_kw={"height_ratios": [1.3, 1]}
        )

        # scene
        ax_s.set_xlim(-300, R0 * 1.15)
        ax_s.set_ylim(-1.5, 1.5)
        ax_s.set_yticks([])

        ax_s.axhline(0, color="gray", lw=0.3)

        ax_s.plot(0, 0, "s", color="#1f77b4", ms=18, zorder=5)
        ax_s.annotate(
            "RADAR", (0, -1.0),
            ha="center", fontsize=9,
            color="#1f77b4", weight="bold"
        )

        ax_s.plot(tgt_x, 0, "D", color=clr, ms=15, zorder=5)
        ax_s.annotate(
            f"TARGET  {label}\nR = {R_now:.0f} m",
            (tgt_x, 0.65),
            ha="center",
            fontsize=9,
            color=clr
        )

        # outgoing pulses
        for x in x_out:
            if 0 < x < tgt_x:
                alpha = 0.45 * (1 - x / tgt_x)
                ax_s.axvline(
                    x,
                    ymin=0.58,
                    ymax=0.78,
                    color="#1f77b4",
                    alpha=max(alpha, 0.06),
                    lw=2
                )

        # reflected echoes
        for x in x_echo:
            if 0 < x < tgt_x:
                alpha = 0.55 * (x / tgt_x)
                ax_s.axvline(
                    x,
                    ymin=0.22,
                    ymax=0.42,
                    color=clr,
                    alpha=max(alpha, 0.06),
                    lw=2.5
                )

        ax_s.text(
            tgt_x * 0.35,
            1.15,
            "→ outgoing wavefronts (fixed spacing = constant transmit frequency)",
            fontsize=8,
            color="#1f77b4",
            ha="center"
        )

        ax_s.text(
            tgt_x * 0.35,
            -0.75,
            "← reflected echoes (spacing changes naturally due to target motion)",
            fontsize=8,
            color=clr,
            ha="center"
        )

        if vr > 0:
            ax_s.text(
                tgt_x * 0.35,
                -1.05,
                "approaching → echoes compressed → higher frequency",
                fontsize=8,
                color=clr,
                ha="center",
                style="italic"
            )

        elif vr < 0:
            ax_s.text(
                tgt_x * 0.35,
                -1.05,
                "receding → echoes stretched → lower frequency",
                fontsize=8,
                color=clr,
                ha="center",
                style="italic"
            )

        else:
            ax_s.text(
                tgt_x * 0.35,
                -1.05,
                "stationary → no Doppler shift",
                fontsize=8,
                color=clr,
                ha="center",
                style="italic"
            )

        ax_s.set_xlabel("Distance [m]")

        ax_s.set_title(
            f"Wavefronts   |   v = {vr:+.0f} m/s   "
            f"f_d = {fd:.0f} Hz  ({fd/1e3:.2f} kHz)   "
            f"λ = {lam*100:.2f} cm",
            fontsize=10
        )

        # baseband waveform visualization
        f_base = 8.0
        doppler_exag = fd / 200

        t_w = np.linspace(0, 2.0, 3000)

        tx_sig = np.cos(2 * np.pi * f_base * t_w)
        rx_sig = np.cos(2 * np.pi * (f_base + doppler_exag) * t_w)

        ax_w.plot(
            t_w,
            tx_sig,
            color="#1f77b4",
            lw=1.2,
            alpha=0.5,
            label=f"Tx  f₀ = {f0/1e9:.1f} GHz"
        )

        ax_w.plot(
            t_w,
            rx_sig,
            color=clr,
            lw=1.8,
            label="Rx  (Doppler shifted)"
        )

        ax_w.set_xlabel("Time [arb. units]")
        ax_w.set_ylabel("Amplitude")

        ax_w.set_title(
            "Baseband: phase drift between Tx and Rx reveals Doppler shift",
            fontsize=10
        )

        ax_w.legend(loc="upper right", fontsize=7)

        plt.tight_layout()
        plt.show()


for w in [w_v3, w_f3, w_t3]:
    w.observe(draw_doppler, "value")

display(widgets.VBox([w_v3, w_f3, w_t3]), out3)

draw_doppler()

Output()

## 5 — Range & Velocity Dashboard

A pulsed-Doppler radar measures **both** range and velocity simultaneously.
Range comes from the echo delay; velocity from the Doppler shift extracted via FFT across
multiple pulses.

For $N$ pulses at PRF, the velocity resolution is

$$\Delta v = \frac{\lambda \cdot \text{PRF}}{2N}$$

The maximum unambiguous velocity is $v_{\max} = \lambda\,\text{PRF}\,/\,4$.

This cell shows three synchronized panels:
- **Left** — the physical scene with the target and its velocity vector.
- **Centre** — the range profile (matched-filter output) marking the target peak.
- **Right** — the Doppler spectrum showing the velocity peak.

In [7]:
out4 = widgets.Output()

w_R4 = widgets.FloatSlider(value=35, min=5, max=100, step=1,
    description="Range [km]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))
w_v4 = widgets.FloatSlider(value=180, min=-400, max=400, step=5,
    description="Velocity [m/s]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))
w_prf4 = widgets.FloatSlider(value=2000, min=500, max=10000, step=100,
    description="PRF [Hz]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))
w_fc4 = widgets.FloatSlider(value=10, min=1, max=35, step=0.5,
    description="Carrier [GHz]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))

def draw_dashboard(change=None):
    R = w_R4.value * 1e3
    vr = w_v4.value
    prf = w_prf4.value
    f0 = w_fc4.value * 1e9
    lam = c / f0
    pri = 1 / prf
    R_ua = c * pri / 2
    v_ua = lam * prf / 4
    fd = 2 * vr * f0 / c
    N = 32
    B = 2e6
    delta_r = c / (2 * B)
    delta_v = lam * prf / (2 * N)

    # range profile
    r_bins = np.linspace(0, R_ua, 500)
    rp = np.exp(-0.5 * ((r_bins - R) / (delta_r / 2)) ** 2)
    noise_r = np.random.default_rng(42).normal(0, 0.04, len(r_bins))
    rp += np.abs(noise_r)

    # doppler spectrum
    v_bins = np.linspace(-v_ua, v_ua, 256)
    dp = np.exp(-0.5 * ((v_bins - vr) / (delta_v / 2)) ** 2)
    noise_v = np.random.default_rng(7).normal(0, 0.03, len(v_bins))
    dp += np.abs(noise_v)

    ambig_r = R > R_ua
    ambig_v = abs(vr) > v_ua

    with out4:
        clear_output(wait=True)
        fig, (ax_s, ax_r, ax_d) = plt.subplots(1, 3, figsize=(14, 4.2))

        # scene
        ax_s.set_xlim(-R_ua * 0.05, R_ua * 1.1)
        ax_s.set_ylim(-1.5, 1.5)
        ax_s.set_yticks([])
        ax_s.axhline(0, color="gray", lw=0.3)
        ax_s.plot(0, 0, "s", color="#1f77b4", ms=16, zorder=5)
        ax_s.annotate("RADAR", (0, -1.0), ha="center",
                      fontsize=8, color="#1f77b4", weight="bold")
        tgt_x = np.clip(R, 500, R_ua * 1.05)
        clr_t = "#d62728" if ambig_r else "#2ca02c"
        ax_s.plot(tgt_x, 0, "D", color=clr_t, ms=14, zorder=5)
        # velocity arrow
        arr_len = np.clip(vr * 2, -R_ua * 0.2, R_ua * 0.2)
        if abs(arr_len) > 50:
            ax_s.annotate("", xy=(tgt_x - arr_len, 0),
                xytext=(tgt_x, 0),
                arrowprops=dict(arrowstyle="->", color="#ff7f0e", lw=2.5))
        ax_s.annotate(f"{R/1e3:.1f} km\n{vr:+.0f} m/s",
                      (tgt_x, 0.6), ha="center", fontsize=8, color=clr_t)
        ax_s.axvline(R_ua, color="gray", ls=":", lw=1)
        ax_s.annotate(f"R_ua\n{R_ua/1e3:.0f} km", (R_ua, 1.2),
                      fontsize=7, ha="center", color="gray")
        ax_s.xaxis.set_major_formatter(
            plt.FuncFormatter(lambda x, _: f"{x/1e3:.0f}"))
        ax_s.set_xlabel("Range [km]")
        warn = "  ⚠ RANGE AMBIGUOUS" if ambig_r else ""
        ax_s.set_title(f"Scene{warn}", fontsize=10, color=clr_t)

        # range profile
        ax_r.plot(r_bins / 1e3, rp, color="#1f77b4", lw=1.5)
        ax_r.fill_between(r_bins / 1e3, rp, alpha=0.2, color="#1f77b4")
        if not ambig_r:
            ax_r.axvline(R / 1e3, color="#d62728", ls="--", lw=1,
                         label=f"R = {R/1e3:.1f} km")
        ax_r.set_xlabel("Range [km]")
        ax_r.set_ylabel("Amplitude")
        ax_r.set_title(f"Range profile  (ΔR = {delta_r:.0f} m)", fontsize=10)
        ax_r.legend(fontsize=7)

        # doppler spectrum
        ax_d.plot(v_bins, dp, color="#ff7f0e", lw=1.5)
        ax_d.fill_between(v_bins, dp, alpha=0.2, color="#ff7f0e")
        if not ambig_v:
            ax_d.axvline(vr, color="#d62728", ls="--", lw=1,
                         label=f"v = {vr:+.0f} m/s")
        ax_d.axvline(v_ua, color="gray", ls=":", lw=0.8)
        ax_d.axvline(-v_ua, color="gray", ls=":", lw=0.8)
        ax_d.set_xlabel("Velocity [m/s]")
        ax_d.set_ylabel("Amplitude")
        warn_v = "  ⚠ VEL AMBIGUOUS" if ambig_v else ""
        ax_d.set_title(f"Doppler spectrum  (Δv = {delta_v:.1f} m/s){warn_v}",
                       fontsize=10, color="#d62728" if ambig_v else "k")
        ax_d.legend(fontsize=7)
        plt.tight_layout()
        plt.show()

for w in [w_R4, w_v4, w_prf4, w_fc4]:
    w.observe(draw_dashboard, "value")
display(widgets.VBox([w_R4, w_v4, w_prf4, w_fc4]), out4)
draw_dashboard()

Output()

## 6 — Range–Doppler Map (2-D Radar Image)

A modern pulse-Doppler radar builds a **Range–Doppler map** by performing an FFT
across the slow-time (pulse) dimension of each range bin. The result is a 2-D image:
range on one axis, velocity on the other, intensity encoding target strength.

Place up to three targets anywhere in the range–velocity plane using the sliders.
The map updates in real time so you can explore how PRF and carrier frequency
reshape the unambiguous window and how close targets merge or separate.

In [ ]:
out5 = widgets.Output()

sl5 = {}
for i, (r0, v0) in enumerate([(30, 100), (60, -200), (45, 50)]):
    sl5[f"r{i}"] = widgets.FloatSlider(value=r0, min=1, max=120, step=1,
        description=f"T{i+1} R [km]:", style={"description_width": "110px"},
        layout=widgets.Layout(width="280px"))
    sl5[f"v{i}"] = widgets.FloatSlider(value=v0, min=-400, max=400, step=5,
        description=f"T{i+1} v [m/s]:", style={"description_width": "110px"},
        layout=widgets.Layout(width="280px"))
w_prf5 = widgets.FloatSlider(value=2000, min=500, max=8000, step=100,
    description="PRF [Hz]:", style={"description_width": "110px"},
    layout=widgets.Layout(width="580px"))
w_fc5 = widgets.FloatSlider(value=10, min=1, max=35, step=0.5,
    description="Carrier [GHz]:", style={"description_width": "110px"},
    layout=widgets.Layout(width="580px"))

def draw_rdmap(change=None):
    prf = w_prf5.value
    f0 = w_fc5.value * 1e9
    lam = c / f0
    pri = 1 / prf
    R_ua = c * pri / 2
    v_ua = lam * prf / 4
    N_dop = 128
    N_rng = 200

    r_ax = np.linspace(0, R_ua, N_rng)
    v_ax = np.linspace(-v_ua, v_ua, N_dop)

    # noise floor
    rng = np.random.default_rng(0)
    rdmap = rng.normal(0, 0.015, (N_rng, N_dop)) ** 2

    # target blobs — use WIDE Gaussians so they are clearly visible
    sigma_r = R_ua / 40
    sigma_v = v_ua / 15
    targets = []
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
    for i in range(3):
        R = sl5[f"r{i}"].value * 1e3
        v = sl5[f"v{i}"].value
        R_fold = R % R_ua
        v_fold_fd = ((2 * v * f0 / c + prf / 2) % prf) - prf / 2
        v_fold = v_fold_fd * c / (2 * f0)
        targets.append((R, v, R_fold, v_fold))
        r_resp = np.exp(-0.5 * ((r_ax - R_fold) / sigma_r) ** 2)
        v_resp = np.exp(-0.5 * ((v_ax - v_fold) / sigma_v) ** 2)
        rdmap += np.outer(r_resp, v_resp)

    with out5:
        clear_output(wait=True)
        fig, (ax_map, ax_sc) = plt.subplots(1, 2, figsize=(13, 5.2),
            gridspec_kw={"width_ratios": [1.4, 1]})

        im = ax_map.imshow(rdmap.T, aspect="auto", origin="lower",
            extent=[0, R_ua / 1e3, -v_ua, v_ua],
            cmap="viridis", vmin=0, vmax=rdmap.max())
        for i, (R, v, Rf, vf) in enumerate(targets):
            ax_map.plot(Rf / 1e3, vf, "x", color=colors[i], ms=14,
                        mew=2.5, label=f"T{i+1} ({R/1e3:.0f}km, {v:+.0f}m/s)")
        ax_map.set_xlabel("Range [km]")
        ax_map.set_ylabel("Velocity [m/s]")
        ax_map.set_title(f"Range–Doppler Map   R_ua={R_ua/1e3:.0f}km   "
                         f"v_ua=±{v_ua:.0f}m/s", fontsize=10)
        ax_map.legend(fontsize=7, loc="upper right")
        fig.colorbar(im, ax=ax_map, shrink=0.7, label="Intensity")

        # scene top view
        ax_sc.set_xlim(-R_ua * 0.05, R_ua * 1.1)
        ax_sc.set_ylim(-R_ua * 0.35, R_ua * 0.35)
        ax_sc.plot(0, 0, "s", color="#1f77b4", ms=14, zorder=5)
        ax_sc.annotate("RADAR", (0, -R_ua * 0.07), ha="center",
                       fontsize=8, color="#1f77b4", weight="bold")
        for i, (R, v, Rf, vf) in enumerate(targets):
            y_off = (i - 1) * R_ua * 0.12
            ax_sc.plot(np.clip(R, 0, R_ua * 1.05), y_off, "D",
                       color=colors[i], ms=12, zorder=5)
            folded = "(folded)" if R > R_ua or abs(v) > v_ua else ""
            ax_sc.annotate(f"T{i+1} {R/1e3:.0f}km {v:+.0f}m/s {folded}",
                (np.clip(R, 0, R_ua * 1.05), y_off + R_ua * 0.04),
                fontsize=7, ha="center", color=colors[i])
            arr_l = np.clip(v * 5, -R_ua * 0.15, R_ua * 0.15)
            if abs(arr_l) > 100:
                ax_sc.annotate("", xy=(np.clip(R, 0, R_ua) - arr_l, y_off),
                    xytext=(np.clip(R, 0, R_ua), y_off),
                    arrowprops=dict(arrowstyle="->", color=colors[i], lw=2))
        ax_sc.axvline(R_ua, color="gray", ls=":", lw=1)
        ax_sc.annotate(f"R_ua", (R_ua, R_ua * 0.3), fontsize=8, color="gray", ha="center")
        ax_sc.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e3:.0f}"))
        ax_sc.set_xlabel("Range [km]")
        ax_sc.set_title("Top-view scene", fontsize=10)
        plt.tight_layout()
        plt.show()

tgt_rows = [widgets.HBox([sl5[f"r{i}"], sl5[f"v{i}"]]) for i in range(3)]
all_sl5 = list(sl5.values()) + [w_prf5, w_fc5]
for w in all_sl5:
    w.observe(draw_rdmap, "value")
display(widgets.VBox(tgt_rows + [w_prf5, w_fc5]), out5)
draw_rdmap()

## 7 — PPI Sweep with Echo Animation

The **Plan Position Indicator** (PPI) is the iconic circular radar screen.
The antenna rotates and a beam sweeps 360°; targets inside the beam produce echoes
that appear as bright dots. Older detections fade like phosphor afterglow.

The beamwidth $\theta_{3\text{dB}} \approx 0.886\,\lambda / D$ determines
the angular resolution — a wider aperture $D$ gives a narrower beam and sharper dots.

Rotate the antenna with the slider and observe how targets drift.

In [ ]:
out6 = widgets.Output()

w_az6 = widgets.FloatSlider(value=0, min=0, max=355, step=5,
    description="Antenna az [°]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))
w_rmax6 = widgets.FloatSlider(value=100, min=20, max=200, step=10,
    description="Display R [km]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))
w_D6 = widgets.FloatSlider(value=3, min=0.5, max=10, step=0.5,
    description="Aperture D [m]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))

np.random.seed(42)
_nt6 = 10
_tr6 = np.random.uniform(10, 90, _nt6)
_taz6 = np.random.uniform(0, 360, _nt6)
_tv6 = np.random.uniform(-1.5, 1.5, _nt6)

def draw_ppi(change=None):
    az = w_az6.value
    R_max = w_rmax6.value
    D = w_D6.value
    f0 = 10e9
    lam = c / f0
    bw_deg = np.rad2deg(0.886 * lam / D)
    tgt_az = (_taz6 + _tv6 * az / 5) % 360

    with out6:
        clear_output(wait=True)
        fig = plt.figure(figsize=(7, 7.5))
        ax = fig.add_subplot(111, projection="polar")
        ax.set_theta_zero_location("N")
        ax.set_theta_direction(-1)
        ax.set_ylim(0, R_max)

        # beam cone
        bw_rad = np.deg2rad(bw_deg)
        beam_az = np.deg2rad(az)
        th_fill = np.linspace(beam_az - bw_rad / 2,
                              beam_az + bw_rad / 2, 50)
        ax.fill_between(th_fill, 0, R_max, alpha=0.15, color="lime")
        ax.plot([beam_az, beam_az], [0, R_max],
                color="lime", lw=1.5, alpha=0.6)

        # phosphor trail
        trail = 16
        for sb in range(trail):
            old_az = az - sb * 5
            old_tgt_az = (_taz6 + _tv6 * old_az / 5) % 360
            diff = np.abs((old_tgt_az - old_az + 180) % 360 - 180)
            det = diff < bw_deg / 2
            alpha = 0.6 * (1 - sb / trail)
            if np.any(det):
                ax.plot(np.deg2rad(old_tgt_az[det]), _tr6[det],
                        "o", color="lime", ms=3.5,
                        alpha=max(alpha, 0.05))

        # current detections
        az_diff = np.abs((tgt_az - az + 180) % 360 - 180)
        det_now = az_diff < bw_deg / 2
        if np.any(det_now):
            ax.plot(np.deg2rad(tgt_az[det_now]), _tr6[det_now],
                    "o", color="lime", ms=8, alpha=0.95)

        ax.set_facecolor("#001a00")
        fig.patch.set_facecolor("#0a0a0a")
        ax.tick_params(colors="lime")
        ax.grid(color="lime", alpha=0.12)
        ax.spines["polar"].set_color("lime")
        ax.set_title(f"PPI Display   θ_3dB = {bw_deg:.1f}°   "
                     f"D = {D:.1f} m",
                     va="bottom", fontsize=11, color="lime", pad=15)
        plt.tight_layout()
        plt.show()

for w in [w_az6, w_rmax6, w_D6]:
    w.observe(draw_ppi, "value")
display(widgets.VBox([w_az6, w_rmax6, w_D6]), out6)
draw_ppi()

## 8 — Ambiguity in Action: Folded Range and Velocity

The PRF simultaneously limits the maximum unambiguous range and velocity:

$$R_{\max} = \frac{c}{2\,\text{PRF}}, \qquad v_{\max} = \frac{\lambda\,\text{PRF}}{4}$$

These two bounds are **inversely coupled** through PRF.
If the target's true range exceeds $R_{\max}$, its echo arrives during the *next* PRI
and appears at a **folded** (wrong) range. Similarly, if the target's Doppler exceeds $\pm v_{\max}$,
the velocity **wraps** into the wrong bin.

This cell plots the **apparent** (folded) range and velocity as you drag the true values
across the ambiguity boundaries. The red zones show where folding occurs.

In [ ]:
out7 = widgets.Output()

w_Rt = widgets.FloatSlider(value=60, min=5, max=300, step=1,
    description="True R [km]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))
w_vt = widgets.FloatSlider(value=100, min=-800, max=800, step=5,
    description="True v [m/s]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))
w_prf7 = widgets.FloatSlider(value=1500, min=200, max=8000, step=100,
    description="PRF [Hz]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))
w_fc7 = widgets.FloatSlider(value=10, min=1, max=35, step=0.5,
    description="Carrier [GHz]:", style={"description_width": "120px"},
    layout=widgets.Layout(width="580px"))

def draw_ambiguity(change=None):
    R_true = w_Rt.value * 1e3
    v_true = w_vt.value
    prf = w_prf7.value
    f0 = w_fc7.value * 1e9
    lam = c / f0
    pri = 1 / prf
    R_ua = c * pri / 2
    v_ua = lam * prf / 4

    R_app = R_true % R_ua
    fd_true = 2 * v_true * f0 / c
    fd_app = ((fd_true + prf / 2) % prf) - prf / 2
    v_app = fd_app * c / (2 * f0)

    folded_r = R_true > R_ua
    folded_v = abs(v_true) > v_ua

    with out7:
        clear_output(wait=True)
        fig, (ax_r, ax_v) = plt.subplots(1, 2, figsize=(13, 4.5))

        # range
        r_arr = np.linspace(0, 300e3, 1000)
        r_app_arr = r_arr % R_ua
        ax_r.plot(r_arr / 1e3, r_app_arr / 1e3, color="#1f77b4", lw=1.5)
        for k in range(1, int(300e3 // R_ua) + 2):
            ax_r.axvline(k * R_ua / 1e3, color="#d62728", ls=":",
                         lw=0.8, alpha=0.5)
        ax_r.plot(R_true / 1e3, R_app / 1e3, "o", color="#d62728" if folded_r else "#2ca02c",
                  ms=12, zorder=5)
        ax_r.annotate(f"True = {R_true/1e3:.0f} km\nAppears at {R_app/1e3:.1f} km",
            (R_true / 1e3, R_app / 1e3 + R_ua / 1e3 * 0.15),
            fontsize=9, ha="center",
            color="#d62728" if folded_r else "#2ca02c",
            bbox=dict(fc="white", ec="gray", alpha=0.8, pad=2))
        ax_r.set_xlabel("True range [km]")
        ax_r.set_ylabel("Apparent range [km]")
        ax_r.set_title(f"Range folding  (R_ua = {R_ua/1e3:.1f} km)"
                       + ("  ⚠ FOLDED" if folded_r else ""),
                       fontsize=10, color="#d62728" if folded_r else "k")

        # velocity
        v_arr = np.linspace(-800, 800, 1000)
        fd_arr = 2 * v_arr * f0 / c
        fd_app_arr = ((fd_arr + prf / 2) % prf) - prf / 2
        v_app_arr = fd_app_arr * c / (2 * f0)
        ax_v.plot(v_arr, v_app_arr, color="#ff7f0e", lw=1.5)
        ax_v.axvline(v_ua, color="#d62728", ls=":", lw=0.8)
        ax_v.axvline(-v_ua, color="#d62728", ls=":", lw=0.8)
        ax_v.plot(v_true, v_app, "o",
                  color="#d62728" if folded_v else "#2ca02c",
                  ms=12, zorder=5)
        ax_v.annotate(f"True = {v_true:+.0f} m/s\nAppears as {v_app:+.0f} m/s",
            (v_true, v_app + v_ua * 0.15),
            fontsize=9, ha="center",
            color="#d62728" if folded_v else "#2ca02c",
            bbox=dict(fc="white", ec="gray", alpha=0.8, pad=2))
        ax_v.set_xlabel("True velocity [m/s]")
        ax_v.set_ylabel("Apparent velocity [m/s]")
        ax_v.set_title(f"Velocity wrapping  (v_ua = ±{v_ua:.0f} m/s)"
                       + ("  ⚠ WRAPPED" if folded_v else ""),
                       fontsize=10, color="#d62728" if folded_v else "k")
        plt.tight_layout()
        plt.show()

for w in [w_Rt, w_vt, w_prf7, w_fc7]:
    w.observe(draw_ambiguity, "value")
display(widgets.VBox([w_Rt, w_vt, w_prf7, w_fc7]), out7)
draw_ambiguity()